# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakeshkumarkhatri/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
!pip -q install duckdb

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [7]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face")

DuckDB connected to Hugging Face


In [8]:
# Hugging Face dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

print("Dataset path configured")

Dataset path configured


In [9]:
schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""").df()

print(schema["column_name"].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [10]:
check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content,
        COUNT(DISTINCT (
            CAST(report_date AS VARCHAR) || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        )) AS distinct_row_keys
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_dates,distinct_clients,distinct_content,distinct_row_keys
0,9841378,31,55,331437,9841378


In [11]:
date_check = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

date_check

,first_date,last_date,distinct_dates
0,2026-03-01,2026-03-31,31


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

- **Unit of analysis:** One row represents one content item for one client on one reporting date.
- **Verification window:** March 1, 2026 through March 31, 2026.
- **Observed:** March 2026 contains 9,841,378 rows across 31 reporting dates.
- **Grain check:** The combination of `report_date`, `client_hash_id`, and `content_hash_id` has 9,841,378 distinct keys, matching the total row count.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field contract

- **Features:** Past-window observed performance signals: GSC impressions, clicks, average position, and GA4 traffic/engagement measures including sessions, users, engaged sessions, engagement time, traffic-source sessions, AI sessions, and scroll events. These are usable only when measured before the prediction point.
- **Label / proxy:** A future-window content-performance outcome (for example, future decline), defined after the feature window. Future outcome fields are not used as features.
- **Context:** `report_date`, `month`, `client_hash_id`, `content_hash_id`, and the GSC/GA4 availability flags. These are used for time windows, grouping, joins, and interpreting missingness rather than being learned from directly.
- **Excluded:** Any future-window performance measurement used to construct the label, because it would not be known at prediction time and would cause leakage.
- **Missingness observed:** In March 2026, GSC impressions and clicks have no NULLs; average position is NULL when `gsc_data_available` is false. GA4 sessions and related GA4/AI/scroll fields are NULL on the rows where `ga4_data_available` is NULL.

In [13]:
window_check = con.sql(f"""
    SELECT
        month,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(*) AS rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month IN ('2026-03', '2026-04')
    GROUP BY month
    ORDER BY month
""").df()

window_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,first_date,last_date,distinct_dates,rows
0,2026-03,2026-03-01,2026-03-31,31,9841378
1,2026-04,2026-04-01,2026-04-30,30,10424730


In [14]:
overlap_check = con.sql(f"""
    WITH march AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '2026-03'
    ),

    april AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '2026-04'
    )

    SELECT
        COUNT(*) AS march_content_client_pairs,
        COUNT(april.content_hash_id) AS pairs_with_april,
        COUNT(*) - COUNT(april.content_hash_id) AS pairs_without_april,
        ROUND(
            100.0 * COUNT(april.content_hash_id) / COUNT(*),
            2
        ) AS pct_with_april
    FROM march
    LEFT JOIN april
        ON march.client_hash_id = april.client_hash_id
        AND march.content_hash_id = april.content_hash_id
""").df()

overlap_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_content_client_pairs,pairs_with_april,pairs_without_april,pct_with_april
0,331437,331436,1,100.0


In [15]:
missing_pair = con.sql(f"""
    WITH march AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '2026-03'
    ),

    april AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '2026-04'
    )

    SELECT
        march.client_hash_id,
        march.content_hash_id
    FROM march
    LEFT JOIN april
        ON march.client_hash_id = april.client_hash_id
        AND march.content_hash_id = april.content_hash_id
    WHERE april.content_hash_id IS NULL
""").df()

missing_pair

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id
0,client_3ffa76342f366962,content_0be895ebbcaab26d


In [16]:
missing_reason = con.sql(f"""
    WITH march AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '2026-03'
    ),

    april AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '2026-04'
    ),

    missing AS (
        SELECT
            march.client_hash_id,
            march.content_hash_id
        FROM march
        LEFT JOIN april
            ON march.client_hash_id = april.client_hash_id
            AND march.content_hash_id = april.content_hash_id
        WHERE april.content_hash_id IS NULL
    )

    SELECT
        COUNT(*) AS missing_pairs,
        COUNT(DISTINCT m.client_hash_id) AS missing_clients,
        COUNT(DISTINCT m.content_hash_id) AS missing_content
    FROM missing m
""").df()

missing_reason

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,missing_pairs,missing_clients,missing_content
0,1,1,1


In [17]:
missingness = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS gsc_impressions_missing,
        AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS gsc_clicks_missing,
        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS gsc_avg_position_missing,

        AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END) AS ga4_sessions_missing,
        AVG(CASE WHEN ga4_engaged_sessions IS NULL THEN 1.0 ELSE 0 END) AS ga4_engaged_sessions_missing,
        AVG(CASE WHEN ga4_total_engagement_sec IS NULL THEN 1.0 ELSE 0 END) AS ga4_engagement_missing,

        AVG(CASE WHEN sessions_ai IS NULL THEN 1.0 ELSE 0 END) AS sessions_ai_missing,
        AVG(CASE WHEN scroll_events IS NULL THEN 1.0 ELSE 0 END) AS scroll_events_missing

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

missingness

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_impressions_missing,gsc_clicks_missing,gsc_avg_position_missing,ga4_sessions_missing,ga4_engaged_sessions_missing,ga4_engagement_missing,sessions_ai_missing,scroll_events_missing
0,9841378,0.0,0.0,0.633074,0.30674,0.30674,0.30674,0.30674,0.30674


In [18]:
ga4_check = con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS rows,
        AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END)
            AS sessions_missing_rate
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY ga4_data_available
    ORDER BY ga4_data_available
""").df()

ga4_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,rows,sessions_missing_rate
0,False,6408671,0.0
1,True,413966,0.0
2,<NA>,3018741,1.0


In [19]:
gsc_check = con.sql(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS rows,
        AVG(
            CASE
                WHEN gsc_avg_position IS NULL THEN 1.0
                ELSE 0
            END
        ) AS avg_position_missing_rate
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY gsc_data_available
    ORDER BY gsc_data_available
""").df()

gsc_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,rows,avg_position_missing_rate
0,False,6230317,1.0
1,True,3611061,0.0


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
# Query 1 — Verify grain

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            CAST(report_date AS VARCHAR) || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        )) AS distinct_grain_keys,
        COUNT(*) - COUNT(DISTINCT (
            CAST(report_date AS VARCHAR) || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        )) AS duplicate_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("Query 1 — Grain verification")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Grain verification


,total_rows,distinct_grain_keys,duplicate_rows
0,9841378,9841378,0


In [24]:
# Query 2 — Counts, missing values and date window

data_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date) AS distinct_dates,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,

        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS gsc_impressions_nulls,

        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS gsc_clicks_nulls,

        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
        ) AS gsc_avg_position_nulls,

        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS ga4_sessions_nulls

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("Query 2 — Counts, missing values and date window")
data_check

Query 2 — Counts, missing values and date window


,total_rows,distinct_dates,first_date,last_date,gsc_impressions_nulls,gsc_clicks_nulls,gsc_avg_position_nulls,ga4_sessions_nulls
0,9841378,31,2026-03-01,2026-03-31,0,0,6230317,3018741


In [25]:
# Query 3 — Availability verification

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) / COUNT(*),
            2
        ) AS gsc_available_pct,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE ga4_data_available IS TRUE
            ) / COUNT(*),
            2
        ) AS ga4_available_pct

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("Query 3 — Availability using IS TRUE")
availability_check

Query 3 — Availability using IS TRUE


,total_rows,gsc_available_rows,ga4_available_rows,gsc_available_pct,ga4_available_pct
0,9841378,3611061,413966,36.69,4.21


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

- **Unbalanced history:** The March-to-April check found 331,436 of 331,437 March client/content pairs with an April observation. One pair has no April row, so a future-window outcome cannot be calculated for every March observation.
- **Incomplete source coverage:** March does not have complete GSC/GA4 coverage. GSC availability is TRUE for 36.69% of rows and GA4 availability is TRUE for 4.21% of rows. GA4-related fields can therefore be unavailable for a substantial part of the slice.
- **Missing values are not automatically zero:** `gsc_avg_position` is NULL for 6,230,317 March rows, while GSC impressions and clicks have no NULLs. Missingness must therefore be handled according to the field's availability rather than replaced with zero without justification.
- **Time-window limitation:** A future label requires a separate future window. Information from the future window cannot be used as a feature because it would not be available at the prediction moment and would create leakage.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.